# 12 - Glacial Lake Detection

## Objectives
- Detect water bodies from Sentinel-2 imagery using NDWI and MNDWI spectral indices
- Apply morphological filtering to clean water masks
- Filter for glacial lakes based on **per-area elevation bounds** from `config_expanded_study_areas.py`
- Vectorize lake polygons for downstream feature extraction

## Methodology
1. **NDWI** (McFeeters, 1996): (Green − NIR) / (Green + NIR) > 0.2
2. **MNDWI** (Xu, 2006): (Green − SWIR) / (Green + SWIR) > 0.1
3. **Elevation filter** (per area):
   - Tropical Andes (Blanca, Vilcanota, Real, etc.): **3000–6500 m**
   - Patagonia Norte / Sur: **0–3200 m** (glaciers reach sea level)
   - Apolobamba / Carabaya: **3500–6000 m**
4. **Size filter**: Minimum 1000 m² to exclude small ponds

## Scientific Relevance
Multi-temporal Sentinel-2 lake mapping (2017–2025) enables detection of lake growth,
a key GLOF precursor. Combined NDWI + MNDWI reduces false positives from snow/ice.
Per-area elevation parameters allow the same pipeline to work across the full
latitudinal range of the Andes (0°S–54°S).

In [1]:
# --- GLOF PROJECT STANDARD SETUP ---
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
try:
    import geopandas as gpd
except ImportError:
    pass

# ── Rutas ──────────────────────────────────────────────────────────────────
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import importlib
import src.gpu_utils
importlib.reload(src.gpu_utils)

# Fix PROJ: usar proj.db de rasterio (v1.6) en vez del de pyproj (v1.4)
if 'PROJ_LIB' in os.environ:
    del os.environ['PROJ_LIB']
try:
    import rasterio as _rio
    _proj_data = Path(_rio.__file__).parent / 'proj_data'
    if _proj_data.exists():
        os.environ['PROJ_LIB'] = str(_proj_data)
    del _rio, _proj_data
except Exception:
    pass

from src.gpu_utils import GPUConfig, gpu_array, cpu_array
gpu_config = GPUConfig()
print(gpu_config)

GPU_AVAILABLE = gpu_config.has_gpu
CUPY_AVAILABLE = gpu_config.cupy_available

GPU CONFIGURATION
GPU Available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU
Device Count: 1
Memory Total: 4.0 GB
Memory Free: 4.0 GB
CUDA Version: None

Library Support:
  - CuPy:         yes
  - PyTorch CUDA: no
  - Numba CUDA:   yes


## 1. Imports

In [2]:
import time
import json

import rasterio
from rasterio.features import shapes
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling

import geopandas as gpd
from shapely.geometry import shape

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from skimage import morphology as skmorph
from scipy import ndimage
from tqdm import tqdm

from src.gpu_utils import gpu_ndwi, gpu_mndwi

print('All imports successful.')

All imports successful.


## 2. Configuration

In [3]:
DATA_DIR = project_root / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Detection thresholds (literature-based for Tropical Andes)
NDWI_THRESHOLD = 0.2    # McFeeters (1996) — 0.2 balances precision/recall for mountain lakes
MNDWI_THRESHOLD = 0.1   # Xu (2006) — 0.1 recommended for turbid glacial lakes
MIN_LAKE_AREA_M2 = 1000  # 1000 m² minimum (INAIGEM standard)

# Default elevation bounds — overridden per area via config below
MIN_ELEVATION_M = 3000   # lower bound (tropical Andes standard)
MAX_ELEVATION_M = 6500   # upper bound
PIXEL_SIZE_M = 10        # Sentinel-2 10 m bands
MIN_PIXELS = int(MIN_LAKE_AREA_M2 / (PIXEL_SIZE_M ** 2))

# ── Per-area elevation bounds from config ─────────────────────────────────
# Patagonia glaciers reach sea level (0–3200 m); tropical Andes: 3000–6500 m.
# The config file is at project_root/config_expanded_study_areas.py.
try:
    from config_expanded_study_areas import EXPANDED_STUDY_AREAS
    AREA_ELEV_BOUNDS = {
        name: (cfg.get('elev_min_m', MIN_ELEVATION_M),
               cfg.get('elev_max_m', MAX_ELEVATION_M))
        for name, cfg in EXPANDED_STUDY_AREAS.items()
    }
    print(f'Loaded elevation bounds for {len(AREA_ELEV_BOUNDS)} areas from config.')
except ImportError:
    print('WARNING: config_expanded_study_areas not found — using global defaults '
          f'({MIN_ELEVATION_M}–{MAX_ELEVATION_M} m) for all areas.')
    AREA_ELEV_BOUNDS = {}

def get_elev_bounds(area_name):
    """Return (min_elev_m, max_elev_m) for a given study area."""
    return AREA_ELEV_BOUNDS.get(area_name, (MIN_ELEVATION_M, MAX_ELEVATION_M))

# Detect study areas from downloaded Sentinel-2 data
s2_dir = RAW_DIR / 'sentinel2'
STUDY_AREAS = sorted([d.name for d in s2_dir.iterdir() if d.is_dir()]) if s2_dir.exists() else []
print(f'Detected {len(STUDY_AREAS)} study areas: {STUDY_AREAS}')
print(f'NDWI threshold : {NDWI_THRESHOLD}')
print(f'MNDWI threshold: {MNDWI_THRESHOLD}')
print(f'Min lake area  : {MIN_LAKE_AREA_M2} m² ({MIN_PIXELS} pixels)')
print()
print('Elevation bounds per area:')
for a in STUDY_AREAS:
    lo, hi = get_elev_bounds(a)
    print(f'  {a:35s}  {lo:>5}–{hi} m')

Loaded elevation bounds for 15 areas from config.
Detected 15 study areas: ['apolobamba', 'bolivia_cordillera_real', 'bolivia_norte', 'carabaya', 'chile_andes_centrales', 'cordillera_blanca', 'cordillera_central', 'cordillera_huanzo', 'cordillera_huayhuash', 'cordillera_raura', 'cordillera_urubamba', 'cordillera_vilcanota', 'ecuador_antisana', 'patagonia_norte', 'patagonia_sur']
NDWI threshold : 0.2
MNDWI threshold: 0.1
Min lake area  : 1000 m² (10 pixels)

Elevation bounds per area:
  apolobamba                            3800–6100 m
  bolivia_cordillera_real               3500–6400 m
  bolivia_norte                         3500–6500 m
  carabaya                              3500–5800 m
  chile_andes_centrales                 2000–6600 m
  cordillera_blanca                     3000–6500 m
  cordillera_central                    3000–5100 m
  cordillera_huanzo                     1800–5300 m
  cordillera_huayhuash                  3500–6500 m
  cordillera_raura                      370

## 3. Validate Prerequisites

In [4]:
def validate_prerequisites():
    """Check that NB11 UTM DEMs and Sentinel-2 scenes are present."""
    print('Validating prerequisites...')
    issues = []

    for area in STUDY_AREAS:
        dem_path = INTERIM_DIR / 'dem' / f'{area}_dem_utm.tif'
        if not dem_path.exists():
            issues.append(f'Missing UTM DEM (run NB11): {dem_path}')
        else:
            print(f'  DEM OK : {area}')

        s2_area_dir = RAW_DIR / 'sentinel2' / area
        if not s2_area_dir.exists():
            issues.append(f'Missing Sentinel-2 directory: {s2_area_dir}')
        else:
            subdirs = [d for d in s2_area_dir.iterdir() if d.is_dir()]
            if not subdirs:
                issues.append(f'No scene folders in: {s2_area_dir}')
            else:
                print(f'  S-2 OK : {area} ({len(subdirs)} scenes/years)')

    if issues:
        print('\nMISSING PREREQUISITES:')
        for iss in issues:
            print(f'  - {iss}')
        return False

    print('\nAll prerequisites satisfied.')
    return True


prerequisites_ok = validate_prerequisites()

Validating prerequisites...
  DEM OK : apolobamba
  S-2 OK : apolobamba (5 scenes/years)
  DEM OK : bolivia_cordillera_real
  S-2 OK : bolivia_cordillera_real (9 scenes/years)
  DEM OK : bolivia_norte
  S-2 OK : bolivia_norte (9 scenes/years)
  DEM OK : carabaya
  S-2 OK : carabaya (8 scenes/years)
  DEM OK : chile_andes_centrales
  S-2 OK : chile_andes_centrales (9 scenes/years)
  DEM OK : cordillera_blanca
  S-2 OK : cordillera_blanca (9 scenes/years)
  DEM OK : cordillera_central
  S-2 OK : cordillera_central (9 scenes/years)
  DEM OK : cordillera_huanzo
  S-2 OK : cordillera_huanzo (9 scenes/years)
  DEM OK : cordillera_huayhuash
  S-2 OK : cordillera_huayhuash (9 scenes/years)
  DEM OK : cordillera_raura
  S-2 OK : cordillera_raura (9 scenes/years)
  DEM OK : cordillera_urubamba
  S-2 OK : cordillera_urubamba (9 scenes/years)
  DEM OK : cordillera_vilcanota
  S-2 OK : cordillera_vilcanota (9 scenes/years)
  DEM OK : ecuador_antisana
  S-2 OK : ecuador_antisana (9 scenes/years)
  D

## 4. Load Sentinel-2 Bands

In [5]:
def _pick_best_scene(folder_list):
    """
    Choose the most recent scene directory.
    Prefers original S2 product folders over aggregated year_ folders.
    """
    s2_folders = sorted([d for d in folder_list if d.name.startswith('S2')])
    year_folders = sorted([d for d in folder_list if d.name.startswith('year_')])
    other_folders = sorted([d for d in folder_list
                            if not d.name.startswith('S2')
                            and not d.name.startswith('year_')])
    ordered = s2_folders + year_folders + other_folders
    return ordered[-1] if ordered else None


def load_sentinel2_bands(area, scene_folder=None):
    """
    Load Sentinel-2 multispectral bands for a study area.

    Bands loaded: B02 (blue), B03 (green), B04 (red), B08 (NIR),
                  B11 (SWIR-1), B12 (SWIR-2).
    SWIR bands at 20 m resolution are resampled to 10 m.

    Parameters
    ----------
    area : str
    scene_folder : Path or None

    Returns
    -------
    dict with 'bands', 'meta', 'date', 'area', or None on failure.
    """
    area_dir = RAW_DIR / 'sentinel2' / area
    if not area_dir.exists():
        print(f'  Directory not found: {area_dir}')
        return None

    all_dirs = [d for d in area_dir.iterdir() if d.is_dir()]
    if not all_dirs:
        print(f'  No scene directories in {area_dir}')
        return None

    folder = Path(scene_folder) if scene_folder else _pick_best_scene(all_dirs)
    print(f'  Loading scene: {folder.name}')

    band_mapping = {
        'B02': 'blue',
        'B03': 'green',
        'B04': 'red',
        'B08': 'nir',
        'B11': 'swir',
        'B12': 'swir2',
    }
    essential = {'B03', 'B08'}  # minimum for NDWI

    bands = {}
    meta = None
    ref_shape = None

    for band_id, band_name in band_mapping.items():
        tif_files = list(folder.glob(f'*{band_id}*.tif'))
        if not tif_files:
            if band_id in essential:
                print(f'  WARNING: essential band {band_id} not found')
            continue

        try:
            with rasterio.open(tif_files[0]) as src:
                if src.width < 10 or src.height < 10:
                    print(f'  WARNING: {band_id} is corrupt ({src.width}x{src.height})')
                    continue
                if meta is None:
                    meta = src.meta.copy()
                    ref_shape = (src.height, src.width)
                data = src.read(1).astype(np.float32)
                if data.shape != ref_shape:
                    from scipy.ndimage import zoom as _zoom
                    sy = ref_shape[0] / data.shape[0]
                    sx = ref_shape[1] / data.shape[1]
                    data = _zoom(data, (sy, sx), order=1)
                bands[band_name] = data
                print(f'  Loaded {band_name} ({band_id}): shape={data.shape}')
        except Exception as exc:
            print(f'  ERROR loading {band_id}: {exc}')

    if 'green' not in bands or 'nir' not in bands:
        print(f'  ERROR: missing green/nir for {area}')
        return None

    # Extract acquisition date from S2 filename (format: S2x_*_YYYYMMDD*)
    scene_date = folder.name  # default: year folder name
    for tif in folder.glob('S2*.tif'):
        parts = tif.name.split('_')
        for p in parts:
            if len(p) == 15 and p[0].isdigit():  # YYYYMMDDTHHMMSS
                scene_date = p[:8]  # YYYYMMDD
                break
        if scene_date != folder.name:
            break

    return {'bands': bands, 'meta': meta, 'date': scene_date, 'area': area}


def load_all_years(area):
    """
    Load one scene per year for a study area (2017-2025).

    Returns
    -------
    list of dicts (each from load_sentinel2_bands, with 'year' key added).
    """
    area_dir = RAW_DIR / 'sentinel2' / area
    if not area_dir.exists():
        return []

    all_dirs = sorted([d for d in area_dir.iterdir() if d.is_dir()])
    results = []
    seen_years = set()

    for scene_dir in all_dirs:
        name = scene_dir.name
        year = None
        if name.startswith('S2') and len(name) > 15:
            try:
                year = int(name.split('_')[2][:4])
            except (IndexError, ValueError):
                pass
        elif name.startswith('year_'):
            try:
                year = int(name.replace('year_', ''))
            except ValueError:
                pass

        elif name.isdigit() and len(name) == 4:
            # Standard structure: data/raw/sentinel2/area/YYYY/S2*_B0x.tif
            year = int(name)
        if year is None:
            # Last resort: use folder mtime (may be inaccurate)
            import datetime
            year = datetime.datetime.fromtimestamp(scene_dir.stat().st_mtime).year

        if year in seen_years:
            continue
        seen_years.add(year)

        data = load_sentinel2_bands(area, scene_folder=scene_dir)
        if data is not None:
            data['year'] = year
            results.append(data)

    print(f'  Loaded {len(results)} year(s) for {area}: {sorted(seen_years)}')
    return results

## 5. Spectral Water Indices

In [6]:
def calculate_ndwi(green, nir):
    """NDWI = (Green - NIR) / (Green + NIR).  McFeeters (1996)."""
    with np.errstate(divide='ignore', invalid='ignore'):
        result = (green - nir) / (green + nir)
        result = np.where(np.isfinite(result), result, 0.0)
    return result.astype(np.float32)


def calculate_mndwi(green, swir):
    """MNDWI = (Green - SWIR) / (Green + SWIR).  Xu (2006)."""
    with np.errstate(divide='ignore', invalid='ignore'):
        result = (green - swir) / (green + swir)
        result = np.where(np.isfinite(result), result, 0.0)
    return result.astype(np.float32)


def detect_water(bands,
                 ndwi_thresh=NDWI_THRESHOLD,
                 mndwi_thresh=MNDWI_THRESHOLD):
    """
    Compute NDWI and MNDWI and return a combined binary water mask.

    GPU acceleration via src.gpu_utils.gpu_ndwi / gpu_mndwi if available.

    Parameters
    ----------
    bands : dict  (keys: 'green', 'nir', optionally 'swir')
    ndwi_thresh  : float
    mndwi_thresh : float

    Returns
    -------
    dict with keys 'ndwi', 'mndwi' (or None), 'water_mask' (bool ndarray).
    """
    t0 = time.time()
    out = {}

    if 'green' not in bands or 'nir' not in bands:
        raise KeyError("bands must contain 'green' and 'nir'")

    out['ndwi'] = gpu_ndwi(bands['green'], bands['nir'])
    ndwi_water = out['ndwi'] > ndwi_thresh

    if 'swir' in bands:
        out['mndwi'] = gpu_mndwi(bands['green'], bands['swir'])
        mndwi_water = out['mndwi'] > mndwi_thresh
        # OR logic: detect water if EITHER index exceeds its threshold.
        # Literature recommendation for glacial mountain lakes:
        #   Gardelle et al. (2011), Pekel et al. (2016) use OR for higher recall.
        out['water_mask'] = ndwi_water | mndwi_water
    else:
        out['mndwi'] = None
        out['water_mask'] = ndwi_water
        print('  NOTE: SWIR absent, using NDWI-only mask')

    elapsed = time.time() - t0
    mode = 'GPU' if CUPY_AVAILABLE else 'CPU'
    print(f'  Water pixels (raw): {int(np.sum(out["water_mask"])):,}  [{elapsed:.2f}s, {mode}]')
    return out

## 6. Morphological Cleaning

In [7]:
def clean_water_mask(mask_arr, min_size_pixels=MIN_PIXELS):
    """
    Clean binary water mask with morphological operations.

    Pipeline:
      1. Remove small objects (< min_size_pixels)
      2. Fill interior holes (ndimage.binary_fill_holes)
      3. Binary closing with disk(2) – connect nearby water pixels
      4. Binary opening with disk(2) – remove thin bridges / noise
      5. Second small-object removal

    Parameters
    ----------
    mask_arr : np.ndarray (bool or uint8)
    min_size_pixels : int  (default = MIN_LAKE_AREA_M2 / pixel_area)

    Returns
    -------
    np.ndarray of dtype uint8 (0/1)
    """
    t0 = time.time()
    cleaned = mask_arr.astype(bool)

    cleaned = skmorph.remove_small_objects(cleaned, min_size=min_size_pixels)
    cleaned = ndimage.binary_fill_holes(cleaned)

    selem = skmorph.disk(2)
    cleaned = skmorph.binary_closing(cleaned, selem)
    cleaned = skmorph.binary_opening(cleaned, selem)
    cleaned = skmorph.remove_small_objects(cleaned, min_size=min_size_pixels)

    print(f'  Cleaned water pixels: {int(np.sum(cleaned)):,}  [{time.time()-t0:.2f}s]')
    return cleaned.astype(np.uint8)

## 7. Elevation Filtering

In [8]:
def filter_by_elevation(water_mask, dem_path, meta,
                        min_elev=MIN_ELEVATION_M,
                        max_elev=MAX_ELEVATION_M):
    """
    Restrict water mask to pixels within the glacial elevation band.

    The UTM DEM is reprojected + resampled to the Sentinel-2 grid
    using bilinear interpolation (rasterio.warp.reproject).

    Parameters
    ----------
    water_mask : np.ndarray
    dem_path   : Path  (UTM DEM from NB11)
    meta       : dict  (rasterio metadata of the S-2 scene)
    min_elev   : float
    max_elev   : float

    Returns
    -------
    tuple (glacial_mask: np.ndarray uint8, dem_resampled: np.ndarray float32 or None)
    """
    dem_path = Path(dem_path)
    if not dem_path.exists():
        print(f'  WARNING: DEM not found ({dem_path}). Skipping elevation filter.')
        return water_mask.astype(np.uint8), None

    dem_resampled = np.empty(
        (meta['height'], meta['width']), dtype=np.float32
    )

    DEM_NODATA = -9999.0
    with rasterio.open(dem_path) as src:
        src_nodata = src.nodata if src.nodata is not None else DEM_NODATA
        reproject(
            source=rasterio.band(src, 1),
            destination=dem_resampled,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=meta['transform'],
            dst_crs=meta['crs'],
            resampling=Resampling.bilinear,
            src_nodata=src_nodata,
            dst_nodata=DEM_NODATA,
        )

    elev_mask = (dem_resampled >= min_elev) & (dem_resampled <= max_elev) & (dem_resampled != DEM_NODATA)
    glacial_mask = water_mask.astype(bool) & elev_mask

    n_glacial = int(np.sum(glacial_mask))
    print(f'  Glacial-zone pixels ({min_elev}\u2013{max_elev} m): {n_glacial:,}')
    return glacial_mask.astype(np.uint8), dem_resampled

## 8. Vectorize Lakes

In [9]:
def vectorize_lakes(water_mask, meta, min_area_m2=MIN_LAKE_AREA_M2):
    """
    Convert binary water raster to a GeoDataFrame of lake polygons.

    Uses rasterio.features.shapes for raster-to-vector conversion.
    Computes: area_m2, perimeter_m, lake_id, compactness.

    Parameters
    ----------
    water_mask : np.ndarray uint8 (0/1)
    meta       : dict  (rasterio metadata)
    min_area_m2 : float

    Returns
    -------
    GeoDataFrame in the CRS of the Sentinel-2 scene.
    """
    lake_geoms = []
    for geom_dict, val in shapes(
        water_mask.astype(np.int16),
        mask=water_mask.astype(bool),
        transform=meta['transform'],
    ):
        if val == 1:
            lake_geoms.append(shape(geom_dict))

    if not lake_geoms:
        print('  No water polygons found after vectorization.')
        return gpd.GeoDataFrame(columns=['geometry', 'area_m2'], crs=meta['crs'])

    gdf = gpd.GeoDataFrame({'geometry': lake_geoms}, crs=meta['crs'])
    gdf['area_m2'] = gdf.geometry.area
    gdf = gdf[gdf['area_m2'] >= min_area_m2].copy().reset_index(drop=True)

    gdf['perimeter_m'] = gdf.geometry.length
    gdf['lake_id'] = range(1, len(gdf) + 1)

    # Compactness: 4*pi*A / P^2  = 1 for circle, < 1 for elongated shapes
    safe_perim = gdf['perimeter_m'].replace(0, np.nan)
    gdf['compactness'] = (4.0 * np.pi * gdf['area_m2'] / safe_perim**2).clip(0, 1)

    print(f'  Vectorized {len(gdf)} lakes \u2265 {min_area_m2} m\u00b2')
    return gdf

## 9. Elevation Statistics per Lake

In [10]:
def add_elevation_stats(lakes_gdf, dem_path):
    """
    Extract DEM statistics for each lake polygon.

    Uses rasterio.mask.mask to clip DEM to each polygon extent.
    Adds columns: elev_mean, elev_min, elev_max, elev_std.

    Parameters
    ----------
    lakes_gdf : GeoDataFrame  (must be in same CRS as DEM)
    dem_path  : Path

    Returns
    -------
    GeoDataFrame with elevation columns added.
    """
    dem_path = Path(dem_path)
    nan_row = {'elev_mean': np.nan, 'elev_min': np.nan,
               'elev_max': np.nan, 'elev_std': np.nan}

    if not dem_path.exists() or len(lakes_gdf) == 0:
        for col, val in nan_row.items():
            lakes_gdf[col] = val
        return lakes_gdf

    records = []
    with rasterio.open(dem_path) as src:
        nodata = src.nodata if src.nodata is not None else -9999
        for _, row in tqdm(lakes_gdf.iterrows(),
                           total=len(lakes_gdf),
                           desc='  Elevation stats'):
            try:
                out_img, _ = mask(src, [row.geometry], crop=True, nodata=nodata)
                valid = out_img[out_img != nodata].astype(float)
                if len(valid) > 0:
                    records.append({
                        'elev_mean': float(np.mean(valid)),
                        'elev_min': float(np.min(valid)),
                        'elev_max': float(np.max(valid)),
                        'elev_std': float(np.std(valid)),
                    })
                else:
                    records.append(nan_row.copy())
            except Exception:
                records.append(nan_row.copy())

    elev_df = pd.DataFrame(records, index=lakes_gdf.index)
    for col in elev_df.columns:
        lakes_gdf[col] = elev_df[col]
    return lakes_gdf

## 10. Complete Detection Pipeline

In [11]:
def detect_glacial_lakes_for_area(area, year_data):
    """
    Full lake detection pipeline for one area and one year.

    Steps:
      1. detect_water  (NDWI + MNDWI)
      2. filter_by_elevation  (per-area bounds from config)
      3. clean_water_mask
      4. vectorize_lakes
      5. add_elevation_stats
      6. Save per-area GeoPackage

    Elevation bounds are read from AREA_ELEV_BOUNDS (populated from
    config_expanded_study_areas.py).  Patagonia areas use 0–3200 m;
    tropical Andes areas use 3000–6500 m.

    Parameters
    ----------
    area      : str
    year_data : dict  (output of load_sentinel2_bands with 'year' key)

    Returns
    -------
    GeoDataFrame (may be empty if no lakes detected).
    """
    year = year_data.get('year', 'unknown')
    elev_min, elev_max = get_elev_bounds(area)

    print(f"\n{'='*60}")
    print(f'  {area}  |  year: {year}  |  elev: {elev_min}–{elev_max} m')
    print('='*60)

    bands = year_data['bands']
    meta = year_data['meta']

    # 1. Water indices
    print('[1/5] Computing water indices...')
    water_results = detect_water(bands)

    # 2. Elevation filter FIRST — applied before morphological ops so that
    #    isolated high-altitude lake pixels are not lost to binary opening.
    print('[2/5] Elevation filtering...')
    dem_path = INTERIM_DIR / 'dem' / f'{area}_dem_utm.tif'
    elev_filtered, _ = filter_by_elevation(
        water_results['water_mask'], dem_path, meta,
        min_elev=elev_min, max_elev=elev_max,
    )

    # 3. Morphological cleaning within the glacial elevation band
    print('[3/5] Morphological cleaning...')
    glacial_mask = clean_water_mask(elev_filtered, min_size_pixels=MIN_PIXELS)

    # 4. Vectorize
    print('[4/5] Vectorizing lakes...')
    lakes_gdf = vectorize_lakes(glacial_mask, meta)

    if len(lakes_gdf) == 0:
        print('  No lakes detected for this area/year.')
        return lakes_gdf

    # 5. Elevation statistics – reproject lakes to DEM CRS if needed
    print('[5/5] Extracting elevation statistics...')
    if dem_path.exists():
        with rasterio.open(dem_path) as src:
            dem_crs = src.crs
        if str(lakes_gdf.crs) != str(dem_crs):
            lakes_gdf = lakes_gdf.to_crs(dem_crs)
    lakes_gdf = add_elevation_stats(lakes_gdf, dem_path)

    # Metadata columns
    lakes_gdf['area_name'] = area
    lakes_gdf['year'] = year
    lakes_gdf['scene_date'] = year_data.get('date', str(year))

    # Save per-area GeoPackage
    out_dir = PROCESSED_DIR / 'lakes'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{area}_{year}.gpkg'
    lakes_gdf.to_file(out_path, driver='GPKG')
    print(f'  Saved {len(lakes_gdf)} lakes -> {out_path}')
    return lakes_gdf

## 11. Main Processing Loop

In [12]:
all_lakes = []

for area in STUDY_AREAS:
    print(f'\nLoading Sentinel-2 for: {area}')
    year_datasets = load_all_years(area)

    if not year_datasets:
        print(f'  No data found for {area}, skipping.')
        continue

    for yd in year_datasets:
        gdf = detect_glacial_lakes_for_area(area, yd)
        if len(gdf) > 0:
            all_lakes.append(gdf)

# ─── Combine to single WGS-84 GeoDataFrame ────────────────────────────────
if all_lakes:
    lakes_wgs84 = [gdf.to_crs('EPSG:4326') for gdf in all_lakes]
    combined = gpd.GeoDataFrame(
        pd.concat(lakes_wgs84, ignore_index=True), crs='EPSG:4326'
    )

    out_combined = PROCESSED_DIR / 'lakes' / 'all_glacial_lakes.gpkg'
    combined.to_file(out_combined, driver='GPKG')

    print(f"\n{'='*60}")
    print('SUMMARY')
    print('='*60)
    print(f'Total lakes detected : {len(combined)}')
    print(f'Total lake area      : {combined["area_m2"].sum()/1e6:.3f} km\u00b2')
    print(f'Study areas          : {combined["area_name"].nunique()}')
    if 'area_name' in combined.columns:
        print('\nPer-area counts:')
        print(combined.groupby('area_name')['area_m2'].count().to_string())
    print(f'\nSaved to: {out_combined}')
else:
    print('WARNING: No lakes detected in any study area.')
    combined = gpd.GeoDataFrame(columns=['geometry', 'area_m2'], crs='EPSG:4326')


Loading Sentinel-2 for: apolobamba
  Loading scene: 2017


  Loaded blue (B02): shape=(9167, 7461)


  Loaded green (B03): shape=(9167, 7461)


  Loaded red (B04): shape=(9167, 7461)


  Loaded nir (B08): shape=(9167, 7461)


...

  Vectorized 1 lakes ≥ 1000 m²
[5/5] Extracting elevation statistics...


  Elevation stats:   0%|          | 0/1 [00:00<?, ?it/s]

  Elevation stats: 100%|██████████| 1/1 [00:00<00:00, 34.39it/s]

  Saved 1 lakes -> D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\lakes\patagonia_sur_2025.gpkg



SUMMARY
Total lakes detected : 12700
Total lake area      : 7064.559 km²
Study areas          : 13

Per-area counts:
area_name
apolobamba                    9
bolivia_cordillera_real     108
carabaya                     87
chile_andes_centrales      1559
cordillera_blanca          3083
cordillera_central         2239
cordillera_huanzo           496
cordillera_huayhuash        278
cordillera_raura           1361
cordillera_urubamba         430
cordillera_vilcanota       2770
ecuador_antisana            272
patagonia_sur                 8

Saved to: D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\lakes\all_glacial_lakes.gpkg


## 12. Visualization

In [13]:
def plot_lake_detection(area, lakes_gdf_area, s2_data, water_results):
    """
    Four-panel detection figure:
    True colour | NDWI | MNDWI | Detected lakes.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    bands = s2_data['bands']

    def to_display(arr, scale=3000.0):
        return np.clip(arr / scale, 0, 1)

    # True colour
    ax = axes[0, 0]
    if all(k in bands for k in ('red', 'green', 'blue')):
        rgb = np.dstack([
            to_display(bands['red']),
            to_display(bands['green']),
            to_display(bands['blue']),
        ])
        ax.imshow(rgb)
    ax.set_title('True Colour (B04/B03/B02)', fontweight='bold')
    ax.axis('off')

    # NDWI
    ax = axes[0, 1]
    if water_results.get('ndwi') is not None:
        im = ax.imshow(water_results['ndwi'], cmap='RdYlBu', vmin=-0.5, vmax=0.5)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title('NDWI (McFeeters 1996)', fontweight='bold')
    ax.axis('off')

    # MNDWI
    ax = axes[1, 0]
    if water_results.get('mndwi') is not None:
        im = ax.imshow(water_results['mndwi'], cmap='RdYlBu', vmin=-0.5, vmax=0.5)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title('MNDWI (Xu 2006)', fontweight='bold')
    ax.axis('off')

    # Detected lakes
    ax = axes[1, 1]
    if all(k in bands for k in ('red', 'green', 'blue')):
        ax.imshow(rgb)
    if len(lakes_gdf_area) > 0:
        lakes_gdf_area.plot(ax=ax, facecolor='cyan', edgecolor='blue',
                            linewidth=0.8, alpha=0.65)
    ax.set_title(f'Detected Glacial Lakes (n={len(lakes_gdf_area)})',
                 fontweight='bold')
    ax.axis('off')

    fig.suptitle(
        f'Lake Detection \u2014 {area.replace("_", " ").title()}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()

    fig_dir = project_root / 'figures'
    fig_dir.mkdir(exist_ok=True)
    out_fig = fig_dir / f'{area}_lake_detection.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figure saved: {out_fig}')


if STUDY_AREAS and all_lakes:
    first_area = STUDY_AREAS[0]
    print(f'Generating detection figure for: {first_area}')
    _s2 = load_sentinel2_bands(first_area)
    if _s2 is not None:
        _wr = detect_water(_s2['bands'])
        _area_lakes_list = [gdf for gdf in all_lakes
                            if 'area_name' in gdf.columns
                            and gdf['area_name'].iloc[0] == first_area]
        _al = _area_lakes_list[0] if _area_lakes_list else gpd.GeoDataFrame(
            columns=['geometry'], crs='EPSG:4326'
        )
        plot_lake_detection(first_area, _al, _s2, _wr)
else:
    print('No data available for visualization.')

Generating detection figure for: apolobamba
  Loading scene: 2024


  Loaded blue (B02): shape=(8419, 7461)


  Loaded green (B03): shape=(8419, 7461)


  Loaded red (B04): shape=(8419, 7461)


  Loaded nir (B08): shape=(8419, 7461)


  Loaded swir (B11): shape=(8419, 7461)


  Loaded swir2 (B12): shape=(8419, 7461)


  Water pixels (raw): 52,781  [0.66s, GPU]


Figure saved: D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\figures\apolobamba_lake_detection.png


## 13. Final Validation

In [14]:
out_file = PROCESSED_DIR / 'lakes' / 'all_glacial_lakes.gpkg'

if out_file.exists():
    check = gpd.read_file(out_file)
    print(f'Output file : {out_file}')
    print(f'Total lakes : {len(check)}')
    print(f'Columns     : {check.columns.tolist()}')
    print(f'CRS         : {check.crs}')
    if 'area_name' in check.columns:
        print('\nLakes per area:')
        print(check['area_name'].value_counts().to_string())
else:
    print(f'WARNING: output file not found: {out_file}')
    print('Check that STUDY_AREAS were processed correctly above.')

Output file : D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\lakes\all_glacial_lakes.gpkg
Total lakes : 12700
Columns     : ['area_m2', 'perimeter_m', 'lake_id', 'compactness', 'elev_mean', 'elev_min', 'elev_max', 'elev_std', 'area_name', 'year', 'scene_date', 'geometry']
CRS         : EPSG:4326

Lakes per area:
area_name
cordillera_blanca          3083
cordillera_vilcanota       2770
cordillera_central         2239
chile_andes_centrales      1559
cordillera_raura           1361
cordillera_huanzo           496
cordillera_urubamba         430
cordillera_huayhuash        278
ecuador_antisana            272
bolivia_cordillera_real     108
carabaya                     87
apolobamba                    9
patagonia_sur                 8
